<img heigth="8" src="https://i.imgur.com/826AqJI.png" alt="pbs-enae">

<h1 align="left">Praparación de datos</h1>

<h2 align="left"><i>Estrategias de marketing basadas en datos</i></h2>

<p align="left">
  <h3>Joseph F. Vergel-Becerra | Introducción a Python - Tools and Skill Courses</h3>
  <br>
  <b>Last updated:</b> <i>23/02/2023</i>
  <br><br>
  <!-- <a href="#tabla-de-contenido">Tabla de contenido</a> • -->
  <a href="#referencias">Referencias</a> •
  <a href="#contribuir">Contribuir</a>
  <!-- <a href="#agradecimientos">Agradecimientos</a> -->
  <br><br>
</p>
<table align="left">
  <td>
      <a href="https://img.shields.io/badge/version-0.1.0-blue.svg?cacheSeconds=2592000">
        <img src="https://img.shields.io/badge/version-0.1.0-blue.svg?cacheSeconds=2592000" alt="Version" height="18">
      </a>
  </td>
  <td>
    <a href="https://colab.research.google.com/github/joefavergel/pbs-enae-python-beginners-course/blob/main/5-python-data-engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
      </a>
  </td>
  <td>
    <a href="https://github.com/joefavergel/pbs-enae-ml-course" target="_parent"><img src="https://img.shields.io/github/forks/joefavergel/pbs-enae-ml-course?style=social" alt="Fork"/>
      </a>
  </td>
</table>
<br>
<br>

---

Ofrecer un servicio o producto a los clientes con un perfil de posible comprador, constituye un factor diferenciador en las estrategias comerciales y de marketing de las compañías e impacta drásticamente en el ahorro de recursos de estas divisiones. Es así como se define el concepto de *target* u objetivo en marketing, que no es más que la definición de los clientes potenciales o público general, que tiene como objetivo una determinada estrategia de marketing, a la hora de hacerles llegar los  productos o servicios de la compañía. En esta oportunidad **representaremos el brazo tecnológico de la división de marketing y comercialización de una institución bancaria** y plantearemos una solución predictiva sobre datos de campañas de marketing telefónico, con la que **predeciremos la probabilidad de que un cliente contrate un certificado de depósito a plazo fijo (CDP)**. De esta manera podremos ofrecerle a nuestros clientes con mayor probabilidad de suscripción, **campañas publicitarias acertadas que maximizen la captación de nuevos contratos** y que al mismo tiempo **reduzcan los costos publicitarios**.

<a id='tabla-de-contenido'></a>
## Tabla de contenido
    
<ol>
    <li><a href="#1-preludio"><b>Preludio</b></a></li>
    <li><a href="#2-ingesta"><b>Ingesta</b></a></li>
    <li><a href="#3-integracion"><b>Integración</b></a></li>
    <li><a href="#4-limpieza"><b>Limpieza: Imputación de valores faltantes</b></a></li>
    <li><a href="#5-transformacion"><b>Transformación</b></a></li>
    <li><a href="#ejercicios"><b>Ejercicios caso práctico: "Preparación de datos"</b></a></li>
</ol>

<a id="1-preludio"></a>
## 1. Preludio

In [ ]:
import sys
from IPython.core.display import HTML

from pathlib import Path
from packaging import version
import sklearn
import urllib.request


print("[INFO] Este proyecto requiere python 3.8 o superior y Scikit-Learn 1.0.1 o superior.")
assert sys.version_info >= (3, 8)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")
print("[INFO] Versiones vificadas exitosamente!")


def css_styling():
    styles_path = Path(f"./styles/custom.css")
    if not styles_path.is_file():
        Path("styles").mkdir(parents=True, exist_ok=True)
        url = f"https://github.com/joefavergel/pbs-enae-ml-course/blob/main/styles/custom.css?raw=true"
        urllib.request.urlretrieve(url, styles_path)

    styles = open("./styles/custom.css", "r").read()
    return HTML(styles)


css_styling()

<a id="2-ingesta"></a>
## 2. Ingesta de datos

La ingesta de datos a través de archivos es una práctica común en la gestión de información. Consiste en la importación de datos desde archivos, ya sean de texto, hojas de cálculo, bases de datos o cualquier otro formato, a un sistema o aplicación para su posterior procesamiento y análisis. Esta técnica es esencial para la mayoría de las empresas y organizaciones, ya que les permite recolectar y gestionar grandes cantidades de información de manera eficiente. En esta ocasión ingestaremos un base de datos de clientes del sector bancario.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import urllib.request

DATASET = "splitted-banking-marketing-targets" # 
DATA_PATH = f"datasets/{DATASET}/"


def download_dataset(dataset: str):
    zipfile_path = Path(f"datasets/{dataset}.zip")
    if not zipfile_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = f"https://github.com/joefavergel/datasets/blob/main/{dataset}.zip?raw=true"
        urllib.request.urlretrieve(url, zipfile_path)
    Path(f"datasets/{dataset}").mkdir(parents=True, exist_ok=True)
    try:
        ZipFile(zipfile_path).extractall(f"datasets/{dataset}")
        print(f"[INFO] Dataset \'{dataset}\' downloaded and uncompressed correctly!")
    except Exception as e:
        print(f"[Exception] There's been a problem: {e}")


download_dataset(dataset=DATASET)

In [ ]:
import os

print(
    f"[INFO] Los archivos presentes en \"{DATA_PATH}\":"
    f"\n\n{os.listdir(DATA_PATH)}"
)

<a id="3-integracion"></a>
## 3. Integración de datos

La etapa de integración de datos se refiere al proceso de combinar datos de diferentes fuentes para crear un conjunto unificado y coherente, adecuado para el análisis o la operación deseada. Este proceso es fundamental en proyectos de ciencia de datos y análisis, donde los datos pueden provenir de múltiples fuentes, formatos y estructuras. El Stack PyData, que incluye herramientas y bibliotecas como **pandas** y  **NumPy**, es invaluable en esta etapa. **Pandas**, en particular, ofrece una amplia variedad de funciones para manipular, transformar y combinar datasets, permitiendo a los usuarios integrar datos con facilidad. Funciones como `merge`, `join` y `concat` en **pandas** facilitan la unión de datasets de diferentes fuentes, mientras que su capacidad para interactuar con múltiples formatos de archivo (como CSV, Excel, SQL y otros) hace que la ingestión y exportación de datos sea un proceso fluido. En conjunto, el Stack PyData proporciona un arsenal robusto para la integración eficiente de datos en proyectos de análisis y ciencia de datos.

In [ ]:
import glob
import numpy as np
import pandas as pd


file_paths = glob.glob("datasets/splitted-banking-marketing-targets/*.csv")

for file_path in file_paths:
    if file_path.endswith('cd.csv'):
        display(pd.read_csv(file_path).T)
    else:
        display(pd.read_csv(file_path))

In [ ]:
cd = pd.read_csv("datasets/splitted-banking-marketing-targets/cd.csv")
cd = cd.T
cd.drop(['Unnamed: 0'], axis=0, inplace=True)
cd = cd.rename(columns={0: 'id', 1: 'cd'})
cd

In [ ]:
personal = pd.read_csv("datasets/splitted-banking-marketing-targets/personal.csv")
personal

In [ ]:
tmp = pd.merge(left=cd, right=personal, how='inner', on='id')
tmp

In [ ]:
history = pd.read_csv(file_paths[3])
history = history.drop(['Unnamed: 0'], axis=1)
history

In [ ]:
pd.merge(left=tmp, right=history, how='inner', on='id')

In [ ]:
pd.concat([cd, personal[['age','job','marital', 'education']]], axis=0)

In [ ]:
# cd = pd.read_csv(file_path).T
file_paths

In [ ]:
pd.DataFrame(personal['id'], columns=['id'])

In [ ]:
import glob

file_paths = glob.glob("datasets/splitted-banking-marketing-targets/*.csv")

def prepare_cd(file_path):
    cd = pd.read_csv(file_path)
    cd = cd.T
    cd.drop(['Unnamed: 0'], axis=0, inplace=True)
    cd = cd.rename(columns={0: 'id', 1: 'cd'})
    return cd



data = pd.DataFrame(personal['id'], columns=['id'])
for file_path in file_paths:
    if file_path.endswith('cd.csv'):
        tmp = prepare_cd(file_path)
    else:
        tmp = pd.read_csv(file_path)
    if 'Unnamed: 0' in list(tmp.columns):
        tmp.drop(['Unnamed: 0'], axis=1, inplace=True, errors='ignore')

    data = pd.merge(left=data, right=tmp, on='id')

data

In [ ]:
data.to_excel('exported.xlsx', index=False)

In [ ]:
data.to_csv('exported.csv', index=False)

In [ ]:
np.ceil(len(data) * 0.1)

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, train_size=0.9, shuffle=True, stratify=data['cd'].values)
train.reset_index(drop=True, inplace=True)
test.reset_index(drop=True, inplace=True)

In [ ]:
data['cd'].value_counts(normalize=True, dropna=False)

In [ ]:
train['cd'].value_counts(normalize=True, dropna=False)

In [ ]:
test['cd'].value_counts(normalize=True, dropna=False)

<a id="4-limpieza"></a>
## 4. Limpieza de datos: Imputación de valores faltantes

La imputación de datos es un proceso de ***reemplazo de valores faltantes*** en conjuntos de datos con valores estimados o predichos. Se utiliza en el análisis de datos para reducir el impacto de los datos faltantes en la precisión de los resultados y para ***garantizar que los datos sean completos y coherentes antes de su análisis***.

Despues de analizar los valore faltantes, se define que:

- `job` y `education` seran imputadas mediante imputacion simple y la estrategia del valor mas frecuente.
- `contact` y `poutcome` seran descartadas del *dataset*.

In [ ]:
!pip install missingno

In [ ]:
import os

import pandas as pd


# train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"), sep=";")
# test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"), sep=";")

train.rename(columns={"y": "cd"}, inplace=True)
test.rename(columns={"y": "cd"}, inplace=True)

with pd.option_context('display.max_columns', None):
    print(f"[INFO] Training dataset dimnesions (rows, cols): {train.shape}")
    display(train.head())
    
    print(f"\n[INFO] testing dataset dimnesions (rows, cols): {test.shape}")
    display(test.head())

In [ ]:
import missingno as msno
import numpy as np


train.replace("unknown", np.nan, inplace=True)
test.replace("unknown", np.nan, inplace=True)

In [ ]:
msno.matrix(train)

In [ ]:
import matplotlib.pyplot as plt
import missingno as msno
from sklearn.impute import SimpleImputer


imp_mode = SimpleImputer(
    missing_values=np.nan,
    strategy='most_frequent'
)
imp_mode.fit(train[["job", "education"]])
train[["job", "education"]] = imp_mode.transform(train[["job", "education"]])

train.drop(["contact", "poutcome"], axis=1, inplace=True)

with pd.option_context('display.max_columns', None):
    display(train)

msno.matrix(train)
plt.show()

Para evitar la fuga de datos o *data leakage*, utilizamos el mismo imputador ajustado en los datos del conjunto de entrenamiento, sobre el conjunto de *test*.

In [ ]:
test[["job", "education"]] = imp_mode.transform(test[["job", "education"]])
test.drop(["contact", "poutcome"], axis=1, inplace=True)

with pd.option_context('display.max_columns', None):
    display(test)

msno.matrix(test)
plt.show()

In [ ]:
import matplotlib.pyplot as plt


plt.hist(train['age'], bins=100)
plt.show()

<a id="i-imputacion"></a>
## 5. Transformación

En cuanto al proceso de ingeniería de características, el gerente comercial y de marketing nos comenta que recientemente ha leído el artículo "[How Age Impacts Consumer Behavior in Retail Banking](https://thefinancialbrand.com/news/financial-education/age-consumer-behavior-patterns-banking-61246/)". El gerente nos explica que en términos generales, el artículo propone un análisis segmentado de cuales son los productos y servicios que más les interesa a los clientes del sector bancario, en términos de  rangos de edad. El gerente manifiesta su interés en llevar este conocimiento al algoritmo de *machine learning*.

<p align="center">
  <a target="_blank">
    <img width="500px" src="https://thefinancialbrand.com/wp-content/uploads/2016/09/financial_education_topics.png" alt="standard-correlation">
  </a>
</p><br><br>

In [ ]:
age_ranges = {
    1: (18, 25),
    2: (25, 35),
    3: (35, 45),
    4: (45, 55),
    5: (55, 65),
    6: (65, 75),
    7: (75, 85),
    8: (85, 200)
}


def transform_age_to_categorical(ages, age_ranges):
    cat_age = [[
        ranges_index for ranges_index, range in age_ranges.items()
        if range[0] <= age < range[1]
    ] for age in ages]

    return [item for sublist in cat_age for item in sublist]


train["cat_age"] = transform_age_to_categorical(train["age"].tolist(), age_ranges)
test["cat_age"] = transform_age_to_categorical(test["age"].tolist(), age_ranges)

with pd.option_context('display.max_columns', None):
    display(train)

In [ ]:
for feature in ["default", "housing", "loan", "cd"]:
    train[feature] = train[feature].replace(["yes", "no"], [1, 0]).astype(int)
    test[feature] = test[feature].replace(["yes", "no"], [1, 0]).astype(int)

train["pdays"] = train["pdays"].replace(-1, 0)
test["pdays"] = test["pdays"].replace(-1, 0)

Necesitamos transformar las características categóricas. Las dos formas basicas para hacerlo son *labeling encoding* y el *one-hot encoding* (variante del *dummy encoding*).

- `job`: Usaremos *labeling encoding* porque hay muchas categorías.
- `education`: Usaremos la codificación de etiquetas porque es una característica categórica ordinal.
- `marital`: Usaremos *one-hot encoding*.
- `month`: Usaremos *one-hot encoding*.
<!-- - `poutcome` : Usaremos *one-hot encoding*. -->

In [ ]:
from sklearn.preprocessing import LabelEncoder


laenc = LabelEncoder()
# train['job'] = laenc.fit_transform(train['job'])
train['education'] = laenc.fit_transform(train['education'])
test['education'] = laenc.fit_transform(test['education'])
train = pd.concat([
    train,
    pd.get_dummies(train["job"], prefix='job'),
    pd.get_dummies(train["marital"], prefix='marital'),
    pd.get_dummies(train["month"], prefix='month')
], axis=1)

test = pd.concat([
    test,
    pd.get_dummies(test["job"], prefix='job'),
    pd.get_dummies(test["marital"], prefix='marital'),
    pd.get_dummies(test["month"], prefix='month')
], axis=1)

with pd.option_context('display.max_columns', None):
    print(f"[INFO] Training dataset dimnesions (rows, cols): {train.shape}")
    display(train.head())

In [ ]:
train["month"] = train["month"].replace(
    list(train["month"].value_counts().index),
    [5, 7, 8, 6, 11, 4, 2, 1, 10, 9, 3, 12]
)
test["month"] = test["month"].replace(
    list(test["month"].value_counts().index),
    [5, 7, 8, 6, 11, 4, 2, 1, 10, 9, 3, 12]
)

train["year"] = pd.Series([2022] * len(train))
train["date"] = pd.to_datetime(train[['year', 'month', 'day']])

test["year"] = pd.Series([2023] * len(test))
test["date"] = pd.to_datetime(test[['year', 'month', 'day']])

with pd.option_context('display.max_columns', None):
    print(f"[INFO] Training dataset dimnesions (rows, cols): {train.shape}")
    display(train.head())
    print(f"\n[INFO] Testing dataset dimnesions (rows, cols): {test.shape}")
    display(test.head())

In [ ]:
!pip install -U feature-engine

`CyclicalFeatures()` aplica transformaciones cíclicas a variables numéricas, devolviendo 2 nuevas características por cada variable, según:

$$var\_sin = \sin(variable * (2. * pi / max\_value))$$

$$var\_cos = \cos(variable * (2. * pi / max\_value))$$

In [ ]:
from feature_engine.creation import CyclicalFeatures


cyc_enc = CyclicalFeatures()
cyc_enc.fit(train[["day", "month"]])
train = pd.concat(
    [train, cyc_enc.transform(train[["day", "month"]])],
axis=1)

test = pd.concat(
    [test, cyc_enc.transform(test[["day", "month"]])],
axis=1)

In [ ]:
plt.figure(figsize=(20, 6))
plt.plot(train['day_sin'][0:48])#.unique())
plt.plot(train['day_cos'][0:48]) #.unique())
plt.xlabel('Time [day]')
plt.title('Time of day signal')
plt.show()

In [ ]:
plt.figure(figsize=(20, 6))
plt.plot(train['month_sin'].unique())
plt.plot(train['month_cos'].unique())
plt.xlabel('Time [day]')
plt.title('Time of day signal')
plt.show()

Efectuamos nuevamente la etapa EDA para generar más observaciones acerca de las características ya transformadas.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


def plot_corr(data: pd.DataFrame, title: str, method: str = "pearson", annot: bool = True):
    f,ax = plt.subplots(figsize=(10,8))
    sns.heatmap(
        data.corr(method=method),
        vmin=-1,
        vmax=1,
        annot=annot,
        linewidths=.5,
        fmt='.2f',
        ax=ax
    )
    plt.title(f"{method.title()}'s Correlation Coefficient between {title}")
    return ax

In [ ]:
ax = plot_corr(
    train.select_dtypes("number"),
    title="the Main Numeric Features",
    method="pearson",
    annot=False
)
plt.show()

In [ ]:
with pd.option_context('display.max_columns', None):
    display(train.select_dtypes("number"))

**Reducción y selección de caracteristicas**

In [ ]:
with pd.option_context('display.max_columns', None):
    display(train)

In [ ]:
baseline_features = [
       'age', 'education', 'default', 'balance', 'housing', 'loan', 'duration', 'campaign',
       'pdays', 'previous', 'job_admin.', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid',
       'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student',
       'job_technician', 'job_unemployed', 'marital_divorced', 'marital_married',
       'marital_single', 'month_apr', 'month_aug', 'month_dec', 'month_feb', 'month_jan',
       'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep'
]
features = [
       'education', 'default', 'balance', 'housing', 'loan', 'duration', 'campaign',
       'pdays', 'previous', 'job_admin.', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid',
       'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student',
       'job_technician', 'job_unemployed', 'marital_divorced', 'marital_married',
       'marital_single', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'cat_age'
]

target = 'cd'

In [ ]:
ax = plot_corr(
    train[features + [target]],
    title="Features and Targets",
    method="kendall",
    annot=False
)
plt.show()

<a id="ejercicios"></a>
## 6. Ejercicios caso práctico: "Análisis de Datos con Python"

1. Replique (**EN LO QUE SEA POSIBLE**) toda la fase de "Preparación de datos" presentada en este `Jupyter notebook`, para el *dataset* que se descargará y descomprimirá  al ejecutar la siguiente celda de codigo:

In [ ]:
import os
from pathlib import Path
from zipfile import ZipFile
import urllib.request

DATASET = "splitted-titanic-dataset"
DATA_PATH = f"datasets/{DATASET}/"


def load_dataset(dataset: str):
    zipfile_path = Path(f"datasets/{dataset}.zip")
    if not zipfile_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = f"https://github.com/joefavergel/datasets/blob/main/{dataset}.zip?raw=true"
        urllib.request.urlretrieve(url, zipfile_path)
    Path(f"datasets/{dataset}").mkdir(parents=True, exist_ok=True)
    try:
        ZipFile(zipfile_path).extractall(f"datasets/{dataset}")
        print(f"[INFO] Dataset \'{dataset}\' downloaded and uncompressed correctly!")
    except Exception as e:
        print(f"[Exception] There's been a problem: {e}")


load_dataset(dataset=DATASET)

print(
    f"[INFO] Los archivos presentes en \"{DATA_PATH}\":"
    f"\n\n{os.listdir(DATA_PATH)}"
)

<a id='referencias'></a>
## Referencias

[1] Halswanter, T. (2016). *An Introduction to Statistics with Python: With Applications in the Life Sciences.* Springer.

[2] Géron, A. (2022). *Hands-on machine learning with Scikit-Learn, Keras, and TensorFlow.* " O'Reilly Media, Inc.".

[3] VanderPlas, J. (2016). *Python data science handbook: Essential tools for working with data.* " O'Reilly Media, Inc.".

[4] Aggarwal, C. C. (2015). *Data mining: the textbook (Vol. 1)*. New York: springer.

<a id='contribuir'></a>
## Contribuir

<p>Para correcciones, <i>bugs</i> o sugerencias, por favor escribe a <a href="mailto:joefavergel@gmail.com">joefavergel@gmail.com</a> o directamente en el repositorio.</p>